In [ ]:
#pip install torch==2.4.1 torchaudio==2.4.1 torchvision==0.19.1 soundfile gdown

In [2]:
import os
import torch
import torch.nn as nn
import torchaudio
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [ ]:
import gdown
import zipfile

# ID del dataset
file_id = '13kmBx_0jGHyWG7lF28ahEZ6g3sPdNTHz'

# Construir la URL de descarga directa
url = f'https://drive.google.com/uc?id={file_id}'
extract_file = "Dataset.zip"

# Descargar el zip
gdown.download(url, extract_file, quiet=False)

print("ZIP descargado")

# Descomprimir
with zipfile.ZipFile(extract_file, 'r') as zip_ref:
    zip_ref.extractall(os.path.splitext(extract_file)[0])

print("ZIP descomprimido en:", os.path.splitext(extract_file)[0])

In [ ]:
# Modelo Unet 2D para Audio Super Resolution

class AttentionGate(nn.Module):
    """Módulo de atención para ponderar skip connections"""
    def __init__(self, F_g, F_l, F_int):

        super().__init__()

        # Basado en el paper "https://arxiv.org/abs/1804.03999"
        # https://github.com/ozan-oktay/Attention-Gated-Networks
        self.W_g = nn.Conv2d(F_g, F_int, kernel_size=1)
        self.W_x = nn.Conv2d(F_l, F_int, kernel_size=1)
        self.psi = nn.Conv2d(F_int, 1, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, g, x):
        # Interpolar g para que tenga el mismo tamaño que x
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
        # Calcular la atención
        att = self.sigmoid(self.psi(self.relu(self.W_g(g) + self.W_x(x))))

        return x * att


class DilatedBlock(nn.Module):
    """Bloque con capas convolucionales dilatadas para capturar contexto de largo alcance sin perder resolución."""
    def __init__(self, channels):

        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(3,1), dilation=(1,1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(6,2), dilation=(2,2)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(12,4), dilation=(4,4)),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x) + x


class UNetAudio2D(nn.Module):
    """Arquitectura UNet 2D adaptada para audio super resolution."""
    def __init__(self):

        super().__init__()

        # Encoder
        # Entrada: (B, 2, F, T)
        self.enc1 = self.conv_block(2, 32)
        self.pool1 = nn.MaxPool2d((2,2))

        self.enc2 = self.conv_block(32, 64)
        self.pool2 = nn.MaxPool2d((2,2))

        self.enc3 = self.conv_block(64, 128)
        self.pool3 = nn.MaxPool2d((2,2))

        self.enc4 = self.conv_block(128, 256)
        self.pool4 = nn.MaxPool2d((2,2))

        # Bottleneck
        self.bottleneck_conv = self.conv_block(256, 512)
        self.bottleneck_dilated = DilatedBlock(512)

        # Decoder
        self.up4 = self.up_block(512,256)
        self.dec4 = self.conv_block(512,256)

        self.up3 = self.up_block(256,128)
        self.dec3 = self.conv_block(256,128)

        self.up2 = self.up_block(128,64)
        self.dec2 = self.conv_block(128,64)

        self.up1 = self.up_block(64,32)
        self.dec1 = self.conv_block(64,32)

        # Attention gates
        self.att4 = AttentionGate(256,256,128)
        self.att3 = AttentionGate(128,128,64)
        self.att2 = AttentionGate(64,64,32)
        self.att1 = AttentionGate(32,32,16)

        # Output
        self.final = nn.Conv2d(32,2,kernel_size=1)

    def conv_block(self, in_ch, out_ch):
        """Bloque convolucional"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=(7,3), padding=(3,1)),
            nn.GroupNorm(out_ch//4, out_ch),    # GroupNorm para batchs pequeños
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=(7,3), padding=(3,1)),
            nn.GroupNorm(out_ch//4, out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def up_block(self, in_ch, out_ch):
        """Bloque de upsampling en frecuencia y tiempo"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),  # sub-pixel upsampling
            nn.LeakyReLU(0.2, inplace=True)
        )

    def match_size(self, x, ref):
        """Asegura que x tenga el mismo tamaño que ref"""
        if x.shape[-2:] != ref.shape[-2:]:
            x = F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)

        return x

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        # Bottleneck
        b = self.bottleneck_conv(self.pool4(e4))
        b = self.bottleneck_dilated(b)

        # Decoder con skip connections y attention gates
        up4 = self.match_size(self.up4(b), e4)
        d4 = self.dec4(torch.cat([up4, self.att4(up4, e4)], dim=1))

        up3 = self.match_size(self.up3(d4), e3)
        d3 = self.dec3(torch.cat([up3, self.att3(up3, e3)], dim=1))

        up2 = self.match_size(self.up2(d3), e2)
        d2 = self.dec2(torch.cat([up2, self.att2(up2, e2)], dim=1))

        up1 = self.match_size(self.up1(d2), e1)
        d1 = self.dec1(torch.cat([up1, self.att1(up1, e1)], dim=1))

        return self.final(d1) + x

In [ ]:
NFFT = 1024
HOP_LENGTH = 256
SAMPLE_RATE = 44100
FRAGMENT_LENGTH = 65536

class AudioSuperResDataset(Dataset):
    """
    Cargar pares de audio (Alta Calidad - HR y Baja Calidad - LR),
    dividirlos en fragmentos de FRAGMENT_LENGTH muestras y devolver
    sus representaciones STFT con shape (2, F, T).

    hr_dir: Directorio de los archivos en alta calidad (Ground Truth)
    lr_dir: Directorio de los archivos en baja calidad (Input)
    fragment_length: Longitud del fragmento a procesar en muestras
    n_fft: Tamaño de la FFT para el STFT.
    hop_length: Salto entre ventanas STFT.
    """
    def __init__(self, hr_dir, lr_dir, fragment_length=FRAGMENT_LENGTH, n_fft=NFFT, hop_length=HOP_LENGTH):

        self.hr_dir = hr_dir
        self.lr_dir = lr_dir
        self.fragment_length = fragment_length
        self.n_fft = n_fft
        self.hop_length = hop_length

        # POOL_FACTOR = 2^4 = 16 (4 capas de pooling en la UNet 2D)
        self.pool_factor = 16

        # Cache de resamplers
        self._resamplers = {}

        # Ventana para STFT
        self.window = torch.hann_window(self.n_fft)

        # Contar ficheros
        files = [
            f for f in os.listdir(hr_dir)
            if f.endswith('.wav') and os.path.exists(os.path.join(lr_dir, f))
        ]

        # Cargar ficheros y dividir en fragmentos
        self.fragments = []  # lista (hr_fragment, lr_fragment)
        for filename in files:
            hr_path = os.path.join(hr_dir, filename)
            lr_path = os.path.join(lr_dir, filename)

            # Cargar los audios
            waveform_hr, sr_hr = torchaudio.load(hr_path)
            waveform_lr, sr_lr = torchaudio.load(lr_path)

            # Uniformizar a mono
            if waveform_hr.size(0) > 1:
                waveform_hr = waveform_hr.mean(dim=0, keepdim=True)
            if waveform_lr.size(0) > 1:
                waveform_lr = waveform_lr.mean(dim=0, keepdim=True)

            # Resamplear a 44.1 kHz
            if sr_hr != SAMPLE_RATE:
                key = ('hr', sr_hr)
                if key not in self._resamplers:
                    self._resamplers[key] = torchaudio.transforms.Resample(sr_hr, SAMPLE_RATE)
                waveform_hr = self._resamplers[key](waveform_hr)

            if sr_lr != SAMPLE_RATE:
                key = ('lr', sr_lr)
                if key not in self._resamplers:
                    self._resamplers[key] = torchaudio.transforms.Resample(sr_lr, SAMPLE_RATE)
                waveform_lr = self._resamplers[key](waveform_lr)

            # Normalizar a [-1, 1]
            max_val = max(waveform_hr.abs().max(), waveform_lr.abs().max()) + 1e-8
            waveform_hr = waveform_hr / max_val
            waveform_lr = waveform_lr / max_val

            # Dividir en fragmentos de fragment_length muestras
            min_len = min(waveform_hr.size(1), waveform_lr.size(1))
            num_fragments = min_len // self.fragment_length

            for i in range(num_fragments):
                start = i * self.fragment_length
                end = start + self.fragment_length
                frag_hr = waveform_hr[:, start:end]
                frag_lr = waveform_lr[:, start:end]

                # Descartar fragmentos de silencio
                if frag_hr.abs().max() < 1e-4:
                    continue

                self.fragments.append((frag_hr, frag_lr))

        print(f"[Dataset] {len(files)} ficheros: {len(self.fragments)} fragmentos de {FRAGMENT_LENGTH} muestras.")

    def __len__(self):
        return len(self.fragments)

    def _waveform_to_stft(self, waveform):
        """
        Convierte una waveform (1, L) a un tensor STFT de shape (2, F, T)
        donde canal 0 = parte real y canal 1 = parte imaginaria.
        """
        stft = torch.stft(
            waveform.squeeze(0),          #(L,)
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.n_fft,
            window=self.window,
            return_complex=True,
        )  #stft shape: (F, T)

        # Separar parte real e imaginaria
        stft_ri = torch.stack([stft.real, stft.imag], dim=0)
        return stft_ri

    def _normalize_stft(self, stft_ri):
        """
        Log-compresión del STFT para reducir el rango dinámico.
        Preserva el signo: sign(x) * log1p(|x|)
        """
        sign = torch.sign(stft_ri)
        return sign * torch.log1p(torch.abs(stft_ri))

    def _pad_stft_to_pool_factor(self, stft):
        """
        Asegura que tanto F como T sean divisibles por pool_factor.
        Padding con ceros si es necesario.
        """
        _, freq_bins, time_frames = stft.shape

        # Pad frecuencia
        pad_f = (self.pool_factor - (freq_bins % self.pool_factor)) % self.pool_factor
        # Pad tiempo
        pad_t = (self.pool_factor - (time_frames % self.pool_factor)) % self.pool_factor

        if pad_f > 0 or pad_t > 0:
            stft = F.pad(stft, (0, pad_t, 0, pad_f))

        return stft

    def __getitem__(self, idx):
        frag_hr, frag_lr = self.fragments[idx]

        # Calcular STFT de ambas waveforms
        stft_lr = self._waveform_to_stft(frag_lr)
        stft_hr = self._waveform_to_stft(frag_hr)

        # Normalizar STFT con log-compresión para reducir rango dinámico
        stft_lr = self._normalize_stft(stft_lr)
        stft_hr = self._normalize_stft(stft_hr)

        # Asegurar que F y T son divisibles por pool_factor (16)
        stft_lr = self._pad_stft_to_pool_factor(stft_lr)
        stft_hr = self._pad_stft_to_pool_factor(stft_hr)

        # Devolver STFT: Input LR y Target HR, ambos con shape (2, F, T)
        return stft_lr, stft_hr

In [ ]:
# Loss para Audio Super-Resolución
# Este script contiene las funciones de pérdida para entrenar la red neuronal, calculando
# la pérdida en el dominio del tiempo y en el dominio de la frecuencia.

NFFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
SAMPLE_RATE = 44100

def stft_mag(x, n_fft=NFFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, window=None):
    """Calcula el STFT y devuelve la magnitud."""
    device = x.device
    if window is None:
        window = torch.hann_window(win_length)

    stft_res = torch.stft(
        x,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        return_complex=True
    )
    
    return torch.abs(stft_res).to(device)


class SpectralConvergenceLoss(nn.Module):
    """Pérdida de convergencia espectral."""

    def forward(self, x_mag, y_mag):
        return torch.norm(y_mag - x_mag, p="fro") / (torch.norm(y_mag, p="fro") + 1e-8)


class LogMagnitudeLoss(nn.Module):
    """Pérdida de magnitud logarítmica."""

    def forward(self, x_mag, y_mag):
        return F.l1_loss(torch.log(x_mag + 1e-7), torch.log(y_mag + 1e-7))


class STFTLoss(nn.Module):
    """Pérdida STFT."""
    def __init__(self, n_fft=NFFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH):

        super().__init__()

        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length

        self.sc_loss = SpectralConvergenceLoss()
        self.mag_loss = LogMagnitudeLoss()

        self.register_buffer("window", torch.hann_window(win_length))

    def forward(self, x, y):
        x_mag = stft_mag(x, self.n_fft, self.hop_length, self.win_length, self.window)
        y_mag = stft_mag(y, self.n_fft, self.hop_length, self.win_length, self.window)

        sc = self.sc_loss(x_mag, y_mag)
        mag = self.mag_loss(x_mag, y_mag)

        return sc, mag


class MultiResolutionSTFTLoss(nn.Module):
    """Pérdida STFT multi-resolución."""
    def __init__(self):

        super().__init__()

        # Calcular STFT a diferentes resoluciones para capturar diferentes frecuencias
        self.stft_losses = nn.ModuleList([
            STFTLoss(512, 128, 512),
            STFTLoss(1024, 256, 1024),
            STFTLoss(2048, 512, 2048),
        ])

    def forward(self, x, y):
        sc_total = 0
        mag_total = 0

        for loss in self.stft_losses:
            sc, mag = loss(x, y)
            sc_total += sc
            mag_total += mag

        sc_total /= len(self.stft_losses)
        mag_total /= len(self.stft_losses)

        return sc_total, mag_total


class HighFrequencyLoss(nn.Module):
    """Pérdida de alta frecuencia."""
    def __init__(self, sr=SAMPLE_RATE, n_fft=NFFT, fmin=4000):

        super().__init__()

        # Crear máscara para ponderar más las altas frecuencias
        freqs = torch.linspace(0, sr/2, n_fft//2 + 1)
        mask = (freqs >= fmin).float()

        self.register_buffer("mask", mask.view(1, -1, 1))

    def forward(self, pred_mag, target_mag):
        return F.l1_loss(pred_mag * self.mask, target_mag * self.mask)


class MelLoss(nn.Module):
    """Pérdida de mel spectrogram."""
    def __init__(self, sr=SAMPLE_RATE, n_fft=NFFT, n_mels=80):

        super().__init__()

        mel_fb = torchaudio.functional.melscale_fbanks(n_freqs=n_fft//2+1, f_min=0, f_max=sr/2, n_mels=n_mels, sample_rate=sr)
        self.register_buffer("mel_fb", mel_fb)

    def forward(self, pred_mag, tgt_mag):
        # pred_mag/tgt_mag shape: (B, F, T)
        pred_mel = torch.matmul(pred_mag.transpose(1, 2), self.mel_fb)
        tgt_mel  = torch.matmul(tgt_mag.transpose(1, 2), self.mel_fb)
        return F.l1_loss(torch.log(pred_mel + 1e-7), torch.log(tgt_mel + 1e-7))


class CombinedLoss(nn.Module):
    """Pérdida combinada de MRSTFT, HF, Complex y Mel."""
    def __init__(self, lambda_mrstft=1.0, lambda_hf=1.0, lambda_complex=1.0, lambda_mel=1.0):

        super().__init__()

        self.lambda_mrstft = lambda_mrstft
        self.lambda_hf = lambda_hf
        self.lambda_complex = lambda_complex
        self.lambda_mel = lambda_mel

        self.mrstft = MultiResolutionSTFTLoss()
        self.hf_loss = HighFrequencyLoss()
        self.mel_loss = MelLoss()

        self.register_buffer("istft_window", torch.hann_window(NFFT))

    def denormalize(self, stft):
        """Inversa de la log-compresión: sign(x) * (exp(|x|) - 1)."""
        sign = torch.sign(stft)
        return sign * (torch.exp(torch.abs(stft)) - 1)

    def forward(self, pred, target):
        # pred/target shape (B,2,F,T)

        # Descartar bins de frecuencia para coincidir con n_fft // 2 + 1 esperado
        valid_f = self.hf_loss.mask.shape[1]
        pred = pred[:, :, :valid_f, :]
        target = target[:, :, :valid_f, :]

        # Desnormalizar
        pred_denorm = self.denormalize(pred)
        target_denorm = self.denormalize(target)

        # complex L1
        complex_l1 = F.l1_loss(pred, target)

        pred_real = pred_denorm[:,0]
        pred_imag = pred_denorm[:,1]

        tgt_real = target_denorm[:,0]
        tgt_imag = target_denorm[:,1]

        pred_mag = torch.sqrt(pred_real**2 + pred_imag**2 + 1e-8)
        tgt_mag = torch.sqrt(tgt_real**2 + tgt_imag**2 + 1e-8)

        # HF loss
        hf = self.hf_loss(pred_mag, tgt_mag)

        # MRSTFT loss
        device = pred.device
        pred_complex = torch.complex(pred_real, pred_imag)
        tgt_complex = torch.complex(tgt_real, tgt_imag)

        window = self.istft_window()
        pred_audio = torch.istft(pred_complex, n_fft=NFFT, hop_length=HOP_LENGTH, window=window).to(device)
        tgt_audio = torch.istft(tgt_complex, n_fft=NFFT, hop_length=HOP_LENGTH, window=window).to(device)

        sc, mag = self.mrstft(pred_audio, tgt_audio)

        mrstft_loss = sc + mag

        # Mel spectrogram loss
        mel_loss = self.mel_loss(pred_mag, tgt_mag)

        return (self.lambda_mrstft * mrstft_loss + self.lambda_hf * hf + self.lambda_complex * complex_l1 +
                self.lambda_mel * mel_loss) / (self.lambda_mrstft + self.lambda_hf + self.lambda_complex + self.lambda_mel)

In [ ]:
# Script para entrenar la red neuronal
# Este script guardara el mejor modelo en un archivo .pt

TRAIN_HR_DIR = 'Dataset/train/HR'  # Archivos de alta resolución (output de la red)
TRAIN_LR_DIR = 'Dataset/train/LR'  # Archivos de baja resolución (input de la red)
VAL_HR_DIR = 'Dataset/test/HR'      # Archivos de alta resolución para validación
VAL_LR_DIR = 'Dataset/test/LR'      # Archivos de baja resolución para validación
BATCH_SIZE = 32
EPOCHS = 500
LEARNING_RATE = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def evaluate(model, dataloader, criterion, device):
    """Evalúa el modelo en el conjunto de validación y devuelve la pérdida promedio."""
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
    return total_loss / len(dataloader)

def train():
    """Entrenar el modelo"""
    dirs_to_check = [TRAIN_HR_DIR, TRAIN_LR_DIR, VAL_HR_DIR, VAL_LR_DIR]
    for d in dirs_to_check:
        if not os.path.exists(d):
            print(f"Error: directorio no encontrado: {d}")
            return

    # Cargar datasets
    train_dataset = AudioSuperResDataset(TRAIN_HR_DIR, TRAIN_LR_DIR)
    val_dataset = AudioSuperResDataset(VAL_HR_DIR, VAL_LR_DIR)

    if len(train_dataset) == 0 or len(val_dataset) == 0:
        print("Error: El dataset está vacío")
        return

    num_workers = max(0, os.cpu_count()-1)
    print(f"Dataset de entrenamiento cargado: {len(train_dataset)} archivos")
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True)
    print(f"Dataset de validación cargado: {len(val_dataset)} archivos")
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)

    # Inicializar modelo
    model = UNetAudio2D().to(DEVICE)

    # Inicializar Loss y Optimizer
    # Loss combinada de MRSTFT, HF, pérdida compleja y mel spectrogram
    # Optimizer AdamW
    criterion = CombinedLoss(lambda_mrstft = 1.0, lambda_hf = 1.0, lambda_complex = 1.0, lambda_mel = 0.5).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4, betas=(0.9, 0.999)) 

    # Scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    print(f"Iniciando entrenamiento en {DEVICE}...")

    best_val_loss = float('inf')
    patience_earlystop = 50
    epochs_no_improve = 0

    # Bucle de entrenamiento
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for inputs, targets in train_dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            # Forward
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            # Backward
            optimizer.zero_grad(set_to_none=True)
            loss.backward()

            # Gradient clipping para evitar explosión de gradientes
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Actualizar los pesos del modelo
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_dataloader)

        # Evaluar en el conjunto de validación
        val_loss = evaluate(model, val_dataloader, criterion, DEVICE)

        print(f"Epoch [{epoch+1}/{EPOCHS}] Train: {train_loss:.6f} | Val: {val_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.8f}")

        # Scheduler basado en val_loss
        scheduler.step(val_loss)

        # Guardar mejor modelo según val_loss
        if epoch >= 5:  # Solo considerar guardar el modelo después de algunas épocas para evitar guardar modelos muy malos al inicio
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), "unet2D_superres.pt")
                print(f"Mejor modelo guardado con loss: {best_val_loss:.6f}")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience_earlystop:
                    print(f"Early stopping en epoch {epoch+1}")
                    break

    print("Entrenamiento completado")
    print(f"Mejor loss alcanzado: {best_val_loss:.6f}")

In [ ]:
train()